# V2 - Validation train/test plus propre

Ce notebook se concentre uniquement sur l'amelioration V2 : rendre la validation chronologique plus claire.

L'idee est simple : un bon backtest full-sample ne suffit pas. Je veux voir si la regle choisie sur la premiere partie de l'historique continue de marcher sur la partie plus recente.

## Ce que veut dire train/test

Le train est la partie ancienne de l'historique. Le modele y selectionne une configuration : hypothese DRIFT ou FADE, mode VWAP, horizon, fenetre d'attente, etc.

Le test est la partie plus recente. On applique la configuration selectionnee sur le train et on regarde si elle marche encore.

Une action peut donc avoir un full-sample positif mais etre rejetee si la configuration selectionnee sur le train perd de l'argent sur le test.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display, Image, Markdown
except ModuleNotFoundError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image({self.filename})"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "macro-vwap-event-study" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

def load_csv(path):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path)
        print(f"Charge {path.name}: {df.shape[0]} lignes, {df.shape[1]} colonnes")
        return df
    print(f"AVERTISSEMENT: fichier manquant: {path}")
    return None

def display_ticker(ticker):
    if str(ticker).upper() == "LONDON-STRATEGIC-EDGE":
        return "LMB"
    return str(ticker)

def add_display_ticker(df):
    if df is not None and "ticker" in df.columns:
        df = df.copy()
        df["display_ticker"] = df["ticker"].map(display_ticker)
    return df

def show_figures(fig_dir):
    fig_dir = Path(fig_dir)
    pngs = sorted(fig_dir.glob("*.png"))
    if not pngs:
        display(Markdown(f"Aucune figure PNG trouvee dans `{fig_dir}`."))
        return
    for path in pngs:
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path)))

V2_DIR = PROJECT_ROOT / "research_tests" / "test_02_train_test_validation" / "results"
v2_master = add_display_ticker(load_csv(V2_DIR / "csv_outputs" / "single_country_master_summary.csv"))
v2_train_test = add_display_ticker(load_csv(V2_DIR / "csv_outputs" / "single_country_train_test.csv"))

## Controle rapide des donnees V2

Je garde une colonne `display_ticker` pour la presentation. Si le CSV contient `LONDON-STRATEGIC-EDGE`, je l'affiche comme LMB. Je signale aussi les doublons comme IRTC.

In [ ]:
if v2_master is not None:
    bad = v2_master[v2_master["ticker"].astype(str).str.upper().eq("LONDON-STRATEGIC-EDGE")]
    if not bad.empty:
        display(Markdown("`LONDON-STRATEGIC-EDGE` detecte. Dans le notebook, je l'affiche comme LMB car le fichier source correspond a LMB."))
        display(bad[["ticker", "display_ticker", "price_file"]])
    dupes = v2_master[v2_master["ticker"].duplicated(keep=False)]
    if not dupes.empty:
        display(Markdown("Doublons detectes dans le master summary :"))
        display(dupes[["ticker", "display_ticker", "price_file"]])

## Qui passe et qui echoue

Je lis surtout `test_status`, `train_ret_moy_pct` et `test_ret_moy_pct`.

- `PASSED_TEST` : le test reste positif et coherent.
- `WEAK_TEST` : le test reste parfois positif, mais la qualite se degrade fortement.
- `FAILED_TEST` : la configuration selectionnee sur le train perd de l'argent sur le test.

In [ ]:
if v2_master is not None:
    cols = ["display_ticker", "classification", "confidence_score", "test_status", "best_ret_moy_pct", "train_ret_moy_pct", "test_ret_moy_pct", "train_test_explanation", "main_warning"]
    cols = [c for c in cols if c in v2_master.columns]
    display(v2_master.sort_values(["classification", "confidence_score"], ascending=[True, False])[cols])

## Noms rejetes malgre un full-sample positif

A positive full-sample result is not enough. I want to know whether the rule selected on the earlier period still works on the later period. If it does not, I treat the result as likely overfit or regime-dependent.

En pratique, je filtre ici :

```text
best_ret_moy_pct > 0
test_ret_moy_pct <= 0
```

In [ ]:
if v2_master is not None:
    filtered = v2_master[
        (pd.to_numeric(v2_master["best_ret_moy_pct"], errors="coerce") > 0)
        & (pd.to_numeric(v2_master["test_ret_moy_pct"], errors="coerce") <= 0)
    ].copy()
    cols = ["display_ticker", "ticker", "best_ret_moy_pct", "train_ret_moy_pct", "test_ret_moy_pct", "classification", "main_warning"]
    cols = [c for c in cols if c in filtered.columns]
    display(filtered[cols].sort_values("best_ret_moy_pct", ascending=False))

## Lecture par groupe

En V2, SMTC et VIAV sont les plus propres.

RVLV et SAM restent interessants, mais je les considere plus fragiles qu'en V1 parce que la validation chronologique est moins forte.

BOOT, ENVX, KOP, LMND, LUMN et VREX sont rejetes parce que le full-sample est positif mais le test devient negatif. C'est exactement le type de resultat que je veux eviter de sur-interpreter.

## Conclusion V2

La V2 rend l'etude plus honnete. Elle force a distinguer un joli backtest complet d'une regle qui tient vraiment sur une periode plus recente.

La prochaine etape devrait etre une V3 plus concentree sur SMTC, VIAV, RVLV, SAM, REAL et IRTC, avec plus de simulations random et une verification plus serieuse de `limit_touch`.